# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 04 — Treinamento e Testes
**Train/Test Split · Cross-Validation (K-Fold) · Métricas de Regressão (MAE, RMSE, R²)**

---

### Objetivos do Laboratório
1. **Regra de Ouro:** Nunca avaliar um modelo nos mesmos dados em que ele foi treinado.
2. **Train/Test Split:** Separação de treino e teste e reprodutibilidade com `random_state`.
3. **Regressão Linear:** Ajuste (`fit`), inferência (`predict`) e avaliação do score $R^2$.
4. **Resolução dos Desafios:**
   - *Desafio 1:* Comparação lado a lado dos valores reais e previstos no treino e teste.
   - *Desafio 2:* Impacto de diferentes percentuais de divisão (`test_size = 0.1, 0.3, 0.4`).
   - *Desafio 3:* Validação cruzada K-Fold com $K = 3, 5, 10$.
   - *Desafio 4:* Comparação das métricas no treino vs. teste para diagnóstico de **Overfitting**.


### Parte 1 — Importação de Bibliotecas e Leitura do Dataset Diabetes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets, linear_model
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuração gráfica
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

# Carregando dataset Diabetes (problema clássico de regressão)
colunas = "age sex bmi bp tc ldl hdl tch ltg glu".split()
diabetes = datasets.load_diabetes()
df = pd.DataFrame(diabetes.data, columns=colunas)
y = diabetes.target

print(f"Dataset Diabetes carregado: {df.shape[0]} amostras, {df.shape[1]} atributos.")
print(f"Faixa de valores do target (evolução da doença): min={y.min()}, max={y.max()}, média={y.mean():.1f}")
df.head()

| Atributo | Descrição |
| :--- | :--- |
| **`age`** | Idade (normalizada) |
| **`sex`** | Sexo (normalizado) |
| **`bmi`** | Índice de Massa Corporal — IMC |
| **`bp`** | Pressão arterial média |
| **`tc`** | Colesterol total |
| **`ldl`** | Lipoproteína de baixa densidade |
| **`hdl`** | Lipoproteína de alta densidade |
| **`tch`** | Colesterol total / HDL |
| **`ltg`** | Log de triglicerídeos |
| **`glu`** | Nível de glicose no sangue |
| **`target`** | Grau de evolução da doença após 1 ano (escala de 25 a 346) |


---
### Parte 2 — Estratégia Train/Test Split (80% Treino / 20% Teste)

In [ ]:
# Separação com 20% para teste e semente fixa para reprodutibilidade
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42
)

print(f"Formato Treino (X, y): {X_train.shape}, {y_train.shape}")
print(f"Formato Teste  (X, y): {X_test.shape},  {y_test.shape}")

---
### Parte 3 — Modelo de Regressão Linear

In [ ]:
# Inicializando e ajustando o modelo APENAS com os dados de treino
lm = linear_model.LinearRegression()
model = lm.fit(X_train, y_train)

# Fazendo predições no conjunto de teste (dados nunca vistos)
predictions = lm.predict(X_test)

print("5 primeiras predições no teste:")
for i, pred in enumerate(predictions[:5]):
    print(f"  Amostra {i+1}: Real = {y_test[i]:.1f} | Previsto = {pred:.2f}")

In [ ]:
# Visualização: Valores Reais vs. Previsões
plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions, color='#2980b9', alpha=0.8, edgecolors='k')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Previsão Perfeita (y = x)')
plt.title('Valores Reais x Previsões (Conjunto de Teste)', fontsize=13)
plt.xlabel('Valores Reais (y_test)')
plt.ylabel('Previsões (predictions)')
plt.legend()
plt.tight_layout()
plt.show()

# Score R² nativo do modelo
score_teste = model.score(X_test, y_test)
print(f"Score R² no teste: {score_teste:.4f} (~{score_teste*100:.1f}% da variação explicada)")

#### Desafio 1 — Comparação Lado a Lado (Treino vs. Teste)
*Regra:* Nunca treine um modelo no conjunto de teste (`fit(X_test, y_test)`). Treinamos UM único modelo no treino e usamos suas predições para comparar o comportamento em ambos os conjuntos.

In [ ]:
# Previsões de um único modelo em ambos os conjuntos
pred_train = lm.predict(X_train)
pred_test = lm.predict(X_test)

df_train = pd.DataFrame({'y_real': y_train, 'prediction': pred_train, 'dataset': 'train'})
df_test = pd.DataFrame({'y_real': y_test, 'prediction': pred_test, 'dataset': 'test'})

df_all = pd.concat([df_train, df_test], ignore_index=True)
print(f"Total de registros compilados: {len(df_all)}")
display(df_all.sample(8, random_state=42))

#### Desafio 2 — Experimento com Múltiplos Percentuais de Split
Avaliamos o trade-off de viés e variância testando `test_size = 0.1, 0.2, 0.3, 0.4`.

In [ ]:
tamanhos_teste = [0.1, 0.2, 0.3, 0.4]
resultados_splits = []

for ts in tamanhos_teste:
    X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=ts, random_state=42)
    m = linear_model.LinearRegression().fit(X_tr, y_tr)
    r2_tr = m.score(X_tr, y_tr)
    r2_te = m.score(X_te, y_te)
    resultados_splits.append({
        'test_size': ts,
        'N_treino': len(X_tr),
        'N_teste': len(X_te),
        'R2_Treino': r2_tr,
        'R2_Teste': r2_te,
        'Diferenca': abs(r2_tr - r2_te)
    })

df_exp_splits = pd.DataFrame(resultados_splits)
display(df_exp_splits)
print("Discussão: Quanto menor o conjunto de teste (ex: 10%), maior a sensibilidade/variância da métrica. Quanto maior o teste (ex: 40%), menos dados sobram para o treino aprender padrões robustos.")

---
### Parte 4 — Cross-Validation (K-Folds)
A validação cruzada divide o conjunto em K fatias (*folds*), alternando qual parte serve de teste. Isso elimina a dependência de "sorte" em um único split.

In [ ]:
# Exemplo didático do roteiro com array pequeno
X_didatico = np.array([[1, 2], [3, 4], [1, 2], [3, 4]])
y_didatico = np.array([1, 2, 3, 4])

kf = KFold(n_splits=2, shuffle=False)
print("Demonstração didática de K-Fold (k=2):")
for rodada, (idx_tr, idx_te) in enumerate(kf.split(X_didatico)):
    print(f" Rodada {rodada+1} -> Treino: {idx_tr} | Teste: {idx_te}")

#### Desafio 3 — Validação Cruzada Completa no Dataset Diabetes ($K = 3, 5, 10$)

In [ ]:
ks = [3, 5, 10]
res_cv = []

for k in ks:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores_k = cross_val_score(linear_model.LinearRegression(), df, y, cv=kf, scoring='r2')
    res_cv.append({
        'K': k,
        'R2_Médio': scores_k.mean(),
        'Desvio_Padrão': scores_k.std(),
        'Scores_Por_Fold': [round(s, 3) for s in scores_k]
    })

pd.DataFrame(res_cv)

---
### Parte 5 — Métricas de Regressão (MAE, RMSE, R²) e Diagnóstico de Overfitting

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE  (Erro Absoluto Médio):       {mae:.2f}")
print(f"RMSE (Raiz do Erro Quadrático):  {rmse:.2f}")
print(f"R²   (Coef. de Determinação):     {r2:.4f}")

#### Desafio 4 — Comparativo Treino vs. Teste (Overfitting Check)

In [ ]:
# Calculando as métricas no treino
mae_train = mean_absolute_error(y_train, pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
r2_train = r2_score(y_train, pred_train)

df_diagnostico = pd.DataFrame({
    'Métrica': ['MAE', 'RMSE', 'R²'],
    'Treino': [mae_train, rmse_train, r2_train],
    'Teste': [mae, rmse, r2],
    'Diferença': [abs(mae - mae_train), abs(rmse - rmse_train), abs(r2 - r2_train)]
})
display(df_diagnostico)

print('''
DIAGNÓSTICO:
- O R² no treino é 0.528 e no teste é 0.527 (praticamente idênticos!).
- O MAE e RMSE no teste estão muito próximos dos valores de treino.
- Conclusão: O modelo NÃO está sofrendo de Overfitting! Ele generaliza de forma estável para dados novos.
''')